<a href="https://colab.research.google.com/github/EdwinZhanCN/Lab-2/blob/main/02_training_clinic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Datasheet

This lab is using UCI Human Activity Recognition, https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones. It is aimed to established a Human Acitivity Recognition System on edge devices like smartphones. We're using it to predict human acitivities via neural networks.

### Sanity check

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

!wget -q https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip
!unzip -q -o "UCI HAR Dataset.zip"

DATA_DIR = 'UCI HAR Dataset'

def load_inertial_signals(subset='train'):
    signal_dir = os.path.join(DATA_DIR, subset, 'Inertial Signals')

    filenames = [
        'body_acc_x', 'body_acc_y', 'body_acc_z',
        'body_gyro_x', 'body_gyro_y', 'body_gyro_z',
        'total_acc_x', 'total_acc_y', 'total_acc_z'
    ]

    signals = []
    for name in filenames:
        filepath = os.path.join(signal_dir, f"{name}_{subset}.txt")
        channel_data = np.loadtxt(filepath)
        signals.append(channel_data)

    X = np.dstack(signals)

    y_path = os.path.join(DATA_DIR, subset, f"y_{subset}.txt")
    y = np.loadtxt(y_path, dtype=int) - 1

    sub_path = os.path.join(DATA_DIR, subset, f"subject_{subset}.txt")
    subjects = np.loadtxt(sub_path, dtype=int)

    return X, y, subjects

X_train_raw, y_train_raw, subjects_train = load_inertial_signals('train')
X_test, y_test, subjects_test = load_inertial_signals('test')

def perform_sanity_checks(X, y, dataset_name="Train"):
    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")

    has_nan = np.isnan(X).any()
    has_inf = np.isinf(X).any()
    print(f"NaN? {has_nan}")
    print(f"Inf? {has_inf}")

    activity_names = ['Walking', 'Walking Upstairs', 'Walking Downstairs', 'Sitting', 'Standing', 'Laying']
    label_counts = pd.Series(y).value_counts().sort_index()

    for label, count in label_counts.items():
        print(f"Class {label} ({activity_names[label]}): {count} windows")
    print("\n")

perform_sanity_checks(X_train_raw, y_train_raw, "Train")
perform_sanity_checks(X_test, y_test, "Test")

X shape: (7352, 128, 9)
y shape: (7352,)
NaN? False
Inf? False
Class 0 (Walking): 1226 windows
Class 1 (Walking Upstairs): 1073 windows
Class 2 (Walking Downstairs): 986 windows
Class 3 (Sitting): 1286 windows
Class 4 (Standing): 1374 windows
Class 5 (Laying): 1407 windows


X shape: (2947, 128, 9)
y shape: (2947,)
NaN? False
Inf? False
Class 0 (Walking): 496 windows
Class 1 (Walking Upstairs): 471 windows
Class 2 (Walking Downstairs): 420 windows
Class 3 (Sitting): 491 windows
Class 4 (Standing): 532 windows
Class 5 (Laying): 537 windows




## Leakage Audit

In [ ]:
import numpy as np

def create_subject_disjoint_split(X, y, subjects, val_ratio=0.2, seed=42):
    np.random.seed(seed)
    unique_subjects = np.unique(subjects)
    print(f"Total: {len(unique_subjects)} -> {unique_subjects}")
    np.random.shuffle(unique_subjects)
    num_val_subjects = int(len(unique_subjects) * val_ratio)

    val_subjects = unique_subjects[:num_val_subjects]
    train_subjects = unique_subjects[num_val_subjects:]

    train_mask = np.isin(subjects, train_subjects)
    val_mask = np.isin(subjects, val_subjects)
    X_train, y_train, sub_train = X[train_mask], y[train_mask], subjects[train_mask]
    X_val, y_val, sub_val = X[val_mask], y[val_mask], subjects[val_mask]
    train_sub_set = set(sub_train)
    val_sub_set = set(sub_val)
    intersection = train_sub_set.intersection(val_sub_set)

    print(f"Train set: {train_sub_set}")
    print(f"Val set: {val_sub_set}")
    print(f"Intersection: {len(intersection)}")

    print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
    print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")

    return X_train, y_train, X_val, y_val

X_train, y_train, X_val, y_val = create_subject_disjoint_split(X_train_raw, y_train_raw, subjects_train)

Total: 21 -> [ 1  3  5  6  7  8 11 14 15 16 17 19 21 22 23 25 26 27 28 29 30]
Train set: {np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(11), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(19), np.int64(21), np.int64(22), np.int64(23), np.int64(26), np.int64(28), np.int64(29), np.int64(30)}
Val set: {np.int64(27), np.int64(1), np.int64(3), np.int64(25)}
Intersection: 0
X_train: (5879, 128, 9), y_train: (5879,)
X_val: (1473, 128, 9), y_val: (1473,)


## Baseline 0:

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

dummy_clf = DummyClassifier(strategy="most_frequent", random_state=42)
dummy_clf.fit(X_train, y_train)

y_val_pred_dummy = dummy_clf.predict(X_val)

dummy_acc = accuracy_score(y_val, y_val_pred_dummy)
dummy_f1 = f1_score(y_val, y_val_pred_dummy, average='macro')

print(f"Baseline 0, Validation Accuracy: {dummy_acc:.4f}")
print(f"Baseline 0, Validation Macro-F1: {dummy_f1:.4f}")

Baseline 0, Validation Accuracy: 0.1758
Baseline 0, Validation Macro-F1: 0.0498


Baseline 0 serves as a reference model. It utilizes a majority-class prediction strategy with a fixed random seed, it simply outputs the most frequent class found in the training data for every prediction. It achieved a validation accuracy of 0.1758 and a macro-F1 score of 0.0498. Any effective deep learning model must significantly outperform these metrics to demonstrate true learning.

## Baseline 1:

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_seed(42)

# Convert to toch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

# Load the data
batch_size = 64
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

class MLPBaseline(nn.Module):
    def __init__(self, input_size=128*9, num_classes=6):
        super(MLPBaseline, self).__init__()
        self.flatten = nn.Flatten()
        self.network = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLPBaseline().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10
print(f"Training on device: {device}")

train_losses = []
val_losses = []
val_f1s = []

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)

    train_loss /= len(train_loader.dataset)
    train_losses.append(train_loss)
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)

            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)

            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss /= len(val_loader.dataset)
    val_f1 = f1_score(all_labels, all_preds, average='macro')

    val_losses.append(val_loss)
    val_f1s.append(val_f1)

    print(f"Epoch [{epoch+1}/{epochs}]Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}")

Training on device: cuda
Epoch [1/10]Train Loss: 0.6710 | Val Loss: 0.2698 | Val F1: 0.8915
Epoch [2/10]Train Loss: 0.2295 | Val Loss: 0.1606 | Val F1: 0.9432
Epoch [3/10]Train Loss: 0.1473 | Val Loss: 0.1539 | Val F1: 0.9424
Epoch [4/10]Train Loss: 0.1314 | Val Loss: 0.2135 | Val F1: 0.9446
Epoch [5/10]Train Loss: 0.1197 | Val Loss: 0.1062 | Val F1: 0.9713
Epoch [6/10]Train Loss: 0.1097 | Val Loss: 0.1775 | Val F1: 0.9480
Epoch [7/10]Train Loss: 0.1005 | Val Loss: 0.1247 | Val F1: 0.9589
Epoch [8/10]Train Loss: 0.1007 | Val Loss: 0.2168 | Val F1: 0.9529
Epoch [9/10]Train Loss: 0.1023 | Val Loss: 0.2725 | Val F1: 0.9429
Epoch [10/10]Train Loss: 0.1162 | Val Loss: 0.2141 | Val F1: 0.9535
